> **Quick start — click *Run All* — no setup needed.**
> Cached PNG figures are displayed by default (`RERUN = False`).
> Set `RERUN = True` and re-run to regenerate figures from the source scripts.


In [ ]:
import subprocess, sys, pathlib, importlib
from IPython.display import Image, display

# ── locate repository root (search upward for lunar/__init__.py) ──────────
_here = pathlib.Path.cwd()
REPO = None
for _p in [_here, *_here.parents]:
    if (_p / "lunar" / "__init__.py").exists():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError("Cannot find REPO root — run from inside Lunar-V2/")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

FIGS    = REPO / "output" / "figures"
SCRIPTS = REPO / "scripts" / "phase2"


def show_figure(name: str, caption: str = "") -> None:
    """Display a cached PNG from output/figures/."""
    path = FIGS / name
    if not path.exists():
        print(f"[warn] figure not found: {path}")
        return
    display(Image(str(path), width=820))
    if caption:
        from IPython.display import Markdown
        display(Markdown(f"*{caption}*"))


def run_script(rel: str) -> None:
    """Run a Phase-2 script via subprocess, streaming stdout."""
    script = SCRIPTS / rel
    print(f"▶  running {script.relative_to(REPO)} …")
    result = subprocess.run(
        [sys.executable, str(script)],
        cwd=str(REPO),
        capture_output=False,
    )
    if result.returncode != 0:
        print(f"[error] script exited with code {result.returncode}")
    else:
        print("[done]")

print(f"REPO  = {REPO}")
print(f"FIGS  = {FIGS}  (exists: {FIGS.exists()})")
print(f"SCRIPTS = {SCRIPTS}  (exists: {SCRIPTS.exists()})")


# Phase 2 (PSR Shoemaker) — Figs 3, 4, 5

| Fig | Subject | Script | Runtime |
|-----|---------|--------|---------|
| 3 | Surface T(t) driven by ray-traced illumination | `psr_shoemaker/fig3_diurnal.py` | ~5–10 min |
| 4 | T(z) steady-state profile, Hayne vs M&S | `psr_shoemaker/figs45_subsurface.py` | <5 s |
| 5 | ΔT(z) = T_M&S − T_Hayne (1-D proxy for 2-D map) | `psr_shoemaker/figs45_subsurface.py` | <5 s |

> Mirrors upstream `PSRShoemaker/UpdatedModel/heat1DShoemaker.m` using the
> same `shoemakerIllumination.mat` input.


In [ ]:
RERUN = False  # Fig 3 takes ~5-10 min; Figs 4-5 are near-instant


## §1 — Upstream illumination input: `shoemakerIllumination.mat`

**Data provenance**

| Field | Value |
|---|---|
| Source | Zenodo [10.5281/zenodo.12586656](https://doi.org/10.5281/zenodo.12586656) |
| Location | lat = −87.91 °, lon = 45.51 ° (Shoemaker crater floor) |
| Samples | 697 (irregular, covering ~23 months of LRO observations) |
| Content | IR re-emission from walls + scattered visible flux + daveTemp reference |
| Local path | `data/upstream/martinez2021/shoemakerIllumination.mat` |

**Summary statistics**

| Quantity | Value |
|---|---|
| Q_total mean | 0.141 W m⁻² |
| daveTemp mean | 34.2 K |

The `.mat` file is loaded by `lunar.illumination.load_shoemaker_illumination()`
which returns a dict with keys `t_jd`, `Q_visible`, `Q_ir`, `Q_total`,
`T_reference`.


In [ ]:
from lunar.illumination import load_shoemaker_illumination
import numpy as np
import matplotlib.pyplot as plt

sh = load_shoemaker_illumination()
days = np.arange(len(sh["t_jd"]))

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(8, 5), sharex=True, constrained_layout=True
)

ax1.fill_between(days, sh["Q_ir"],
                 alpha=0.5, color="#d62728", label="IR re-emission from walls")
ax1.fill_between(days, sh["Q_visible"],
                 alpha=0.5, color="#1f77b4", label="Scattered visible")
ax1.plot(days, sh["Q_total"], "k-", lw=1.2,
         label=f"Q_total (mean {sh['Q_total'].mean():.3f} W/m²)")
ax1.set_ylabel("Flux (W m⁻²)")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_title("Shoemaker illumination (real ray-traced upstream data)")

ax2.plot(days, sh["T_reference"], color="0.3", lw=1.2,
         marker=".", ms=3,
         label=f"daveTemp (mean {sh['T_reference'].mean():.1f} K)")
ax2.set_xlabel("Sample index (697 over ~23 months)")
ax2.set_ylabel("T reference (K)")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.show()

print(f"Q_total : mean={sh['Q_total'].mean():.4f}, peak={sh['Q_total'].max():.4f} W/m²")
print(f"daveTemp: mean={sh['T_reference'].mean():.2f}, "
      f"range=[{sh['T_reference'].min():.1f}, {sh['T_reference'].max():.1f}] K")


## §2 — Figure 3 · Shoemaker surface T(t), Hayne vs M&S

The Phase-1 solver is driven by `Q_total(t)` from `shoemakerIllumination.mat`,
resampled to **10-minute time steps** over the full 697-sample (~23-month) span.

Key solver settings:
- `spinup_depth_m = 0.10 m` (top-only convergence)
- Convergence tolerance = 0.05 K
- Both Hayne (2017) and Martinez & Siegler (2021) K(T,ρ) models run
  back-to-back; results overlaid against `daveTemp` reference.

> **Faithful replication** — this notebook uses the *identical* `Q(t)` input
> that the upstream MATLAB script `heat1DShoemaker.m` used.


In [ ]:
if RERUN:
    print(
        "Fig 3 takes ~5-10 min "
        "(time-domain spin-up × 2 K models, 697-day span at 10-min steps)."
    )
    run_script("psr_shoemaker/fig3_diurnal.py")

show_figure(
    "phase2_fig3_shoemaker_diurnal.png",
    "Fig 3 — Shoemaker surface T(t) driven by real upstream illumination",
)


## §3 — Figures 4 & 5 · Shoemaker T(z) and ΔT(z)

**Steady-state heat equation** on the geometric depth grid with:
- Dirichlet upper BC: T_surface = daveTemp mean = **34.2 K**
- Lower BC: geothermal flux (Hayne 2017 default)

Figure 4 shows the full T(z) profile for Hayne vs M&S down to 4 m depth.
Figure 5 shows ΔT(z) = T_M&S − T_Hayne, the 1-D proxy for the 2-D
horizontal temperature-difference map (the "4-m depth" result quoted in the
paper).  The exact ΔT(4 m) value will be printed to stdout when
`figs45_subsurface.py` runs.


In [ ]:
if RERUN:
    run_script("psr_shoemaker/figs45_subsurface.py")

show_figure(
    "phase2_fig4_shoemaker_Tz.png",
    "Fig 4 — Shoemaker T(z), Hayne vs M&S",
)
show_figure(
    "phase2_fig5_dT_vs_depth.png",
    "Fig 5 — ΔT vs depth, 1-D proxy for 2-D 4-m map",
)


## §4 — References

1. **Martinez & Siegler (2021)** — *A Study of Lunar Polar Crater Environments
   Using a Thermal Model*, JGR Planets, 126, e2020JE006659.
   <https://doi.org/10.1029/2020JE006659>
2. **Zenodo data archive** — `shoemakerIllumination.mat` and companion files.
   <https://doi.org/10.5281/zenodo.12586656>
3. **Hayne et al. (2017)** — *Global Regolith Thermophysical Properties of the
   Moon from the Diviner Lunar Radiometer*, JGR Planets, 122, 2371–2400.
   <https://doi.org/10.1002/2017JE005303>
4. **Vasavada et al. (2012)** — *Lunar Equatorial Surface Temperatures and
   Regolith Properties from the Diviner Lunar Radiometer*, JGR Planets, 117,
   E00H18. <https://doi.org/10.1029/2011JE003987>
5. **Mazarico et al. (2011)** — *Illumination conditions of the lunar polar
   regions using LOLA topography*, Icarus, 211, 1066–1081.
   <https://doi.org/10.1016/j.icarus.2010.10.030>
6. **Diviner GCP PDS** — Lunar Reconnaissance Orbiter Diviner Lunar
   Radiometer Grid-Cell Product, NASA PDS Geosciences Node.
   <https://pds-geosciences.wustl.edu/lro/lro-l-dlre-5-gcp-v1/>
